# **Importando as Bibliotecas necessárias**

In [11]:
%pip install streamlit
%pip install plotly
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import streamlit as st
import pandas as pd
import plotly.express as px
import seaborn as sns

**Carregando o data set e removendo as colunas indesejadas para as análises**

In [13]:
# Carregar o arquivo com o separador ponto e vírgula
df = pd.read_csv('cantonamento_ferroviario_pt_leve.csv', sep=';')

# Array com colunas para serem removidas
colunas_para_remover = [
    'id_trecho',
    'nome_linha_fr',
    'modo_cantonamento_fr',
    'pk_inicial_orig',
    'pk_final_orig',
    'coord_geo_inicio',
    'coord_geo_fim',
    'geo_ponto_central'
]

# Removendo as colunas
df_formatado = df.drop(columns=[col for col in colunas_para_remover if col in df.columns])

# Exibir as primeiras linhas do DataFrame formatado
display(df_formatado.head())

,codigo_linha,nome_linha,ordem_trecho,modo_cantonamento,familia_tecnologica,nivel_automacao,nivel_automacao_rotulo,tipo_via,pk_inicial_km,pk_final_km,extensao_km,latitude_inicio,longitude_inicio,latitude_fim,longitude_fim,x_inicio_l93,y_inicio_l93,x_fim_l93,y_fim_l93
0,205000,Linha de Soissons a Givet,1,Sem cantonamento,Sem cantonamento,0.0,0 – Sem cantonamento,Não especificado,16.953,23.134,6.181,49.332394,3.541827,49.305320,3.614871,739392.4614,6.914893e+06,744726.3373,6.911920e+06
1,955000,Linha de La Râpée a Batignolles (petite ceintu...,1,Bloco automático luminoso (BAL),Bloco automático,3.0,3 – Automático lateral,Não especificado,21.020,31.974,10.954,48.840868,2.407917,48.895879,2.324570,656543.9415,6.860254e+06,650479.1635,6.866420e+06
2,797306,Ramal de ligação de St-Rambert-d'Albon (bifurc...,1,Bloco automático luminoso (BAL),Bloco automático,3.0,3 – Automático lateral,Não especificado,0.000,1.052,1.052,45.301437,4.803102,45.309119,4.807634,841306.5643,6.468510e+06,841642.1622,6.469371e+06
3,819000,Linha de St-Césaire ao Grau-du-Roi,1,Cantonamento telefônico – via única (CT),Cantonamento telefônico,1.0,1 – Telefônico,Via única,20.260,39.416,19.156,43.700874,4.275361,43.570873,4.191084,802824.2162,6.289924e+06,796248.0038,6.275372e+06
4,753000,Linha de Laroche-Migennes a Cosne,1,Bloco automático luminoso – via banalizada (BA...,Bloco automático,3.0,3 – Automático lateral,Via banalizada (bidirecional),190.398,192.369,1.971,47.685302,3.672682,47.670919,3.677335,750465.6530,6.731872e+06,750828.4056,6.730277e+06


# **Criando o dashboard**

In [14]:
# Configurações iniciais da página do Streamlit
st.set_page_config(page_title="Cantonamento Ferroviário", layout="wide")


# Definições para o tratamento do dataframe no streamlit, com as formatações feitas anteriormente
@st.cache_data
def preparar_dados_dashboard(dataframe_base):
    df_dashboard = dataframe_base.copy()

    # Trata os valores nulos do nível de automação como "Não classificado"
    df_dashboard['nivel_automacao_rotulo'] = df_dashboard['nivel_automacao_rotulo'].fillna('Não classificado').astype(str)

    # Calcula as coordenadas médias de latitude e longitude de cada trecho para plotar no mapa
    df_dashboard['lat_meio'] = (df_dashboard['latitude_inicio'] + df_dashboard['latitude_fim']) / 2
    df_dashboard['lon_meio'] = (df_dashboard['longitude_inicio'] + df_dashboard['longitude_fim']) / 2

    return df_dashboard

# Atribuição dos dados preparados a partir do dataframe global
df = preparar_dados_dashboard(df_formatado)



#1 --- Barra Lateral (Filtros) ---
st.sidebar.header("Filtros")

# Filtro de Família Tecnológica
familias = st.sidebar.multiselect(
    "Família tecnológica",
    sorted(df['familia_tecnologica'].dropna().unique()),
    default=sorted(df['familia_tecnologica'].dropna().unique())
)


# Filtro de Tipo de Via
vias = st.sidebar.multiselect(
    "Tipo de via",
    sorted(df['tipo_via'].dropna().unique()),
    default=sorted(df['tipo_via'].dropna().unique())
)


# Filtro de Nível de Automação
niveis = st.sidebar.multiselect(
    "Nível de automação",
    sorted(df['nivel_automacao_rotulo'].unique()),
    default=sorted(df['nivel_automacao_rotulo'].unique())
)


# Aplicação dos filtros selecionados sobre o DataFrame principal
f = df[
    df['familia_tecnologica'].isin(familias) &
    df['tipo_via'].isin(vias) &
    df['nivel_automacao_rotulo'].isin(niveis)
]


#2 --- Cabeçalho e Métricas Principais ---
st.title("Cantonamento Ferroviário")

# Cálculos matemáticos de extensão geral e automática
km_total = f['extensao_km'].sum()
km_auto = f.loc[f['nivel_automacao'] >= 3, 'extensao_km'].sum()

# Exibição dos principais KPIs organizados em 4 colunas
c1, c2, c3, c4 = st.columns(4)
c1.metric("Trechos", f"{len(f):,}".replace(",", "."))
c2.metric("Linhas", f"{f['codigo_linha'].nunique():,}".replace(",", "."))
c3.metric("Extensão total (km)", f"{km_total:,.0f}".replace(",", "."))
c4.metric("% km automático (nível ≥ 3)", f"{(km_auto / km_total * 100) if km_total else 0:.1f}%")



#3 --- Primeiro Bloco de Gráficos (Família e Modo de Cantonamento) ---
col1, col2 = st.columns(2)


# Gráfico de barras horizontais: Extensão por família tecnológica
por_familia = (f.groupby('familia_tecnologica', as_index=False)['extensao_km']
                 .sum().sort_values('extensao_km'))
col1.plotly_chart(
    px.bar(por_familia, x='extensao_km', y='familia_tecnologica', orientation='h',
           title="Extensão (km) por família tecnológica",
           labels={'extensao_km': 'km', 'familia_tecnologica': ''}),
    use_container_width=True
)


# Gráfico de barras horizontais: Top 10 modos de cantonamento mais comuns
por_modo = (f.groupby('modo_cantonamento', as_index=False)['extensao_km']
              .sum().nlargest(10, 'extensao_km').sort_values('extensao_km'))
col2.plotly_chart(
    px.bar(por_modo, x='extensao_km', y='modo_cantonamento', orientation='h',
           title="Top 10 modos de cantonamento (km)",
           labels={'extensao_km': 'km', 'modo_cantonamento': ''}),
    use_container_width=True
)



#3 --- Segundo Bloco de Gráficos (Tipo de Via e Distribuição da Extensão) ---
col3, col4 = st.columns(2)


# Gráfico empilhado: Relação entre o tipo de via e as famílias tecnológicas
via_fam = (f.groupby(['tipo_via', 'familia_tecnologica'], as_index=False)['extensao_km'].sum())
col3.plotly_chart(
    px.bar(via_fam, x='tipo_via', y='extensao_km', color='familia_tecnologica',
           title="Tipo de via × família tecnológica (km)",
           labels={'extensao_km': 'km', 'tipo_via': ''}),
    use_container_width=True
)


# Histograma: Distribuição das extensões em escala logarítmica
col4.plotly_chart(
    px.histogram(f.dropna(subset=['extensao_km']), x='extensao_km', nbins=60,
                 log_x=True, title="Distribuição da extensão dos trechos (km, escala log)"),
    use_container_width=True
)


# Gráfico de barras horizontais: Maiores ferrovias/linhas por km total
top_linhas = (f.groupby('nome_linha', as_index=False)['extensao_km']
                .sum().nlargest(15, 'extensao_km').sort_values('extensao_km'))
st.plotly_chart(
    px.bar(top_linhas, x='extensao_km', y='nome_linha', orientation='h',
           title="15 maiores linhas por extensão (km)",
           labels={'extensao_km': 'km', 'nome_linha': ''}, height=550),
    use_container_width=True
)



#4 --- Mapa dos Trechos ---
st.subheader("Mapa dos trechos")
st.plotly_chart(
    px.scatter_map(f, lat='lat_meio', lon='lon_meio', color='familia_tecnologica',
                   hover_name='nome_linha',
                   hover_data={'modo_cantonamento': True, 'extensao_km': ':.1f',
                               'lat_meio': False, 'lon_meio': False},
                   zoom=4.5, height=600, map_style='carto-positron'),
    use_container_width=True
)



#5 --- Tabela de Dados Expandível ---
with st.expander("Ver dados filtrados"):
    st.dataframe(f)


2026-11-14 10:18:32.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.738 No runtime found, using MemoryCacheStorageManager
2026-11-14 10:18:32.757 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.758 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.763 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.769 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.771 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 10:18:32.776 Thread 'MainThread':

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px

# Carregamento dos dados
df = pd.read_csv('cantonamento_ferroviario_pt_leve.csv', sep=';')

# Remoção das colunas indesejadas
colunas_para_remover = [
    'id_trecho',
    'nome_linha_fr',
    'modo_cantonamento_fr',
    'pk_inicial_orig',
    'pk_final_orig',
    'coord_geo_inicio',
    'coord_geo_fim',
    'geo_ponto_central'
]

df_formatado = df.drop(
    columns=[col for col in colunas_para_remover if col in df.columns]
)


# Configurações iniciais da página do Streamlit
st.set_page_config(page_title="Cantonamento Ferroviário", layout="wide")


# Definições para o tratamento do dataframe no streamlit, com as formatações feitas anteriormente
@st.cache_data
def preparar_dados_dashboard(dataframe_base):
    df_dashboard = dataframe_base.copy()

    # Trata os valores nulos do nível de automação como "Não classificado"
    df_dashboard['nivel_automacao_rotulo'] = df_dashboard['nivel_automacao_rotulo'].fillna('Não classificado').astype(str)

    # Calcula as coordenadas médias de latitude e longitude de cada trecho para plotar no mapa
    df_dashboard['lat_meio'] = (df_dashboard['latitude_inicio'] + df_dashboard['latitude_fim']) / 2
    df_dashboard['lon_meio'] = (df_dashboard['longitude_inicio'] + df_dashboard['longitude_fim']) / 2

    return df_dashboard

# Atribuição dos dados preparados a partir do dataframe global
df = preparar_dados_dashboard(df_formatado)



#1 --- Barra Lateral (Filtros) ---
st.sidebar.header("Filtros")

# Filtro de Família Tecnológica
familias = st.sidebar.multiselect(
    "Família tecnológica",
    sorted(df['familia_tecnologica'].dropna().unique()),
    default=sorted(df['familia_tecnologica'].dropna().unique())
)


# Filtro de Tipo de Via
vias = st.sidebar.multiselect(
    "Tipo de via",
    sorted(df['tipo_via'].dropna().unique()),
    default=sorted(df['tipo_via'].dropna().unique())
)


# Filtro de Nível de Automação
niveis = st.sidebar.multiselect(
    "Nível de automação",
    sorted(df['nivel_automacao_rotulo'].unique()),
    default=sorted(df['nivel_automacao_rotulo'].unique())
)


# Aplicação dos filtros selecionados sobre o DataFrame principal
f = df[
    df['familia_tecnologica'].isin(familias) &
    df['tipo_via'].isin(vias) &
    df['nivel_automacao_rotulo'].isin(niveis)
]


#2 --- Cabeçalho e Métricas Principais ---
st.title("Cantonamento Ferroviário")

# Cálculos matemáticos de extensão geral e automática
km_total = f['extensao_km'].sum()
km_auto = f.loc[f['nivel_automacao'] >= 3, 'extensao_km'].sum()

# Exibição dos principais KPIs organizados em 4 colunas
c1, c2, c3, c4 = st.columns(4)
c1.metric("Trechos", f"{len(f):,}".replace(",", "."))
c2.metric("Linhas", f"{f['codigo_linha'].nunique():,}".replace(",", "."))
c3.metric("Extensão total (km)", f"{km_total:,.0f}".replace(",", "."))
c4.metric("% km automático (nível ≥ 3)", f"{(km_auto / km_total * 100) if km_total else 0:.1f}%")



#3 --- Primeiro Bloco de Gráficos (Família e Modo de Cantonamento) ---
col1, col2 = st.columns(2)


# Gráfico de barras horizontais: Extensão por família tecnológica
por_familia = (f.groupby('familia_tecnologica', as_index=False)['extensao_km']
                 .sum().sort_values('extensao_km'))
col1.plotly_chart(
    px.bar(por_familia, x='extensao_km', y='familia_tecnologica', orientation='h',
           title="Extensão (km) por família tecnológica",
           labels={'extensao_km': 'km', 'familia_tecnologica': ''}),
    use_container_width=True
)


# Gráfico de barras horizontais: Top 10 modos de cantonamento mais comuns
por_modo = (f.groupby('modo_cantonamento', as_index=False)['extensao_km']
              .sum().nlargest(10, 'extensao_km').sort_values('extensao_km'))
col2.plotly_chart(
    px.bar(por_modo, x='extensao_km', y='modo_cantonamento', orientation='h',
           title="Top 10 modos de cantonamento (km)",
           labels={'extensao_km': 'km', 'modo_cantonamento': ''}),
    use_container_width=True
)



#3 --- Segundo Bloco de Gráficos (Tipo de Via e Distribuição da Extensão) ---
col3, col4 = st.columns(2)


# Gráfico empilhado: Relação entre o tipo de via e as famílias tecnológicas
via_fam = (f.groupby(['tipo_via', 'familia_tecnologica'], as_index=False)['extensao_km'].sum())
col3.plotly_chart(
    px.bar(via_fam, x='tipo_via', y='extensao_km', color='familia_tecnologica',
           title="Tipo de via × família tecnológica (km)",
           labels={'extensao_km': 'km', 'tipo_via': ''}),
    use_container_width=True
)


# Histograma: Distribuição das extensões em escala logarítmica
col4.plotly_chart(
    px.histogram(f.dropna(subset=['extensao_km']), x='extensao_km', nbins=60,
                 log_x=True, title="Distribuição da extensão dos trechos (km, escala log)"),
    use_container_width=True
)


# Gráfico de barras horizontais: Maiores ferrovias/linhas por km total
top_linhas = (f.groupby('nome_linha', as_index=False)['extensao_km']
                .sum().nlargest(15, 'extensao_km').sort_values('extensao_km'))
st.plotly_chart(
    px.bar(top_linhas, x='extensao_km', y='nome_linha', orientation='h',
           title="15 maiores linhas por extensão (km)",
           labels={'extensao_km': 'km', 'nome_linha': ''}, height=550),
    use_container_width=True
)



#4 --- Mapa dos Trechos ---
st.subheader("Mapa dos trechos")
st.plotly_chart(
    px.scatter_map(f, lat='lat_meio', lon='lon_meio', color='familia_tecnologica',
                   hover_name='nome_linha',
                   hover_data={'modo_cantonamento': True, 'extensao_km': ':.1f',
                               'lat_meio': False, 'lon_meio': False},
                   zoom=4.5, height=600, map_style='carto-positron'),
    use_container_width=True
)



#5 --- Tabela de Dados Expandível ---
with st.expander("Ver dados filtrados"):
    st.dataframe(f)


Overwriting app.py


In [ ]:
import sys

!{sys.executable} -m streamlit run app.py

**Pontos de atenção nos dados:**


1. `nivel_automacao` tem 212 nulos, que correspondem a “Outro / não especificado”. Tratado como “Não classificado” em vez de descartar.
2. `tipo_via` é “Não especificado” em 1.011 trechos (62%), então gráficos por tipo de via cobrem só uma parte da rede.
3. `extensao_km` tem 2 nulos (as linhas com PK em formato D+841). Elas ficam de fora dos totais em km.